In [458]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import xgboost as xgb
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import cross_val_score
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier

In [459]:
df = pd.read_csv("reddit_features.csv")

In [460]:
drop_cols = ["viral", "score", "awards", "url", "title", "subreddit"]

In [461]:
x = df.drop(columns=drop_cols)
y = df["viral"]

In [462]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state = 42)

In [463]:
model = xgb.XGBClassifier(scale_pos_weight=3)

In [464]:
model.fit(x_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [465]:
y_pred = model.predict(x_test)

In [466]:
print(accuracy_score(y_test, y_pred))

0.735


In [467]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.83      0.82      0.82       151
           1       0.46      0.47      0.46        49

    accuracy                           0.73       200
   macro avg       0.64      0.65      0.64       200
weighted avg       0.74      0.73      0.74       200



In [468]:
print(confusion_matrix(y_test, y_pred))

[[124  27]
 [ 26  23]]


In [469]:
scores = cross_val_score(model, x, y, cv=5)
print(scores.mean())

0.7310000000000001


In [470]:
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=5
)

In [471]:
model.fit(x_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_parallel_tree=None, ...)

In [472]:
y_prob = model.predict_proba(x_test)[:,1]
y_pred = (y_prob > 0.35).astype(int)

In [473]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.89      0.65      0.75       151
           1       0.41      0.76      0.53        49

    accuracy                           0.68       200
   macro avg       0.65      0.70      0.64       200
weighted avg       0.77      0.68      0.70       200



In [474]:
x.columns

Index(['upvote_ratio', 'num_comments', 'created_utc', 'is_self', 'post_hour',
       'post_day', 'is_weekend', 'title_words_count', 'has_question',
       'has_number', 'controversy_index', 'sentiment_score',
       'is_extreme_sentiment'],
      dtype='object')

In [475]:
model.feature_importances_

array([0.0561594 , 0.17021167, 0.06655246, 0.1091933 , 0.07562184,
       0.08111791, 0.        , 0.09892911, 0.0678992 , 0.03620176,
       0.07767249, 0.08847097, 0.07196995], dtype=float32)

In [476]:
model_rf = RandomForestClassifier(class_weight="balanced")

In [477]:
model_rf.fit(x_train, y_train)

RandomForestClassifier(class_weight='balanced')

In [478]:
y_probrf = model_rf.predict_proba(x_test)[:,1]
y_predrf = (y_probrf > 0.35).astype(int)

In [479]:
print(classification_report(y_test, y_predrf))

              precision    recall  f1-score   support

           0       0.84      0.81      0.83       151
           1       0.48      0.53      0.50        49

    accuracy                           0.74       200
   macro avg       0.66      0.67      0.67       200
weighted avg       0.75      0.74      0.75       200



In [480]:
model_rf.feature_importances_

array([0.0616593 , 0.26233114, 0.10973939, 0.00766964, 0.09529558,
       0.04752181, 0.01120298, 0.12012388, 0.00363151, 0.01901035,
       0.14560077, 0.10349708, 0.01271658])

In [481]:
import joblib
joblib.dump(model, "xgb_model.pkl")

['xgb_model.pkl']